Methods of creating delta table: 

1.Query

In [0]:
%sql
create or replace table retail_products_sql(
    product_id int,
    product_name string,
    price int
)
using delta;

In [0]:
%sql
insert into retail_products_sql values(1, 'Laptop', 50000), (2, 'Mpuse', 800), (3, 'Keyboard',1500);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from retail_products_sql;

product_id,product_name,price
1,Laptop,50000
2,Mpuse,800
3,Keyboard,1500


2.Using dataframe


In [0]:
data = [
    (4, "Monitor", 12000),
    (5, "Printer", 18000),
    (6, "Webcam", 2500)
]
 
df = spark.createDataFrame(
    data,
    ["product_id", "product_name", "price"]
)
 
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_products_pyspark")
 
display(spark.table("retail_products_pyspark"))
 

product_id,product_name,price
4,Monitor,12000
5,Printer,18000
6,Webcam,2500


3. Create table as select (CTAS)

In [0]:
%sql
CREATE OR REPLACE TABLE retail_products_ctas
USING DELTA
AS
SELECT *
FROM VALUES
    (7, 'Tablet', 25000),
    (8, 'Speaker', 5000),
    (9, 'Router', 3500)
AS products(product_id, product_name, price);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM retail_products_ctas;

product_id,product_name,price
7,Tablet,25000
8,Speaker,5000
9,Router,3500


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_updates AS
SELECT *
FROM VALUES
    (2, 'Wireless Mouse', 950),
    (10, 'USB Hub', 1200)
AS updates(product_id, product_name, price);

In [0]:
%sql
MERGE INTO retail_products_sql AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.price = source.price

WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, price)
    VALUES (source.product_id, source.product_name, source.price);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM retail_products_sql
ORDER BY product_id;

product_id,product_name,price
1,Laptop,50000
2,Wireless Mouse,950
3,Keyboard,1500
10,USB Hub,1200


In [0]:
%sql
describe detail retail_products_sql;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e675b90a-ae70-483c-b798-87439a9c6712,dbt.default.retail_products_sql,null,abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/a5e339a0-0baf-48ce-a33a-2e669cb9c673,2026-11-05T07:10:02.377Z,2026-11-05T08:48:03.000Z,List(),List(),1,2202,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-359b3085-b280-46e0-803e-4214a0d0f733, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-fc989ddd-3721-419a-847a-ff478003b0d1)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
describe history retail_products_sql;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-05T08:48:03.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),a3d03bb0-828e-4b06-8e0f-54a1f9f5a68c,1105-070950-djhheq45-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5308, p25FileSize -> 2202, numDeletionVectorsRemoved -> 1, minFileSize -> 2202, numAddedFiles -> 1, maxFileSize -> 2202, p75FileSize -> 2202, p50FileSize -> 2202, numAddedBytes -> 2202)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-05T08:48:01.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(product_id#11263 = product_id#11254)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2163559625184539),a3d03bb0-828e-4b06-8e0f-54a1f9f5a68c,1105-070950-djhheq45-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4005, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5609, materializeSourceTimeMs -> 377, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2493, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2656)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-05T07:11:07.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2163559625184539),e9b06b4b-f003-4673-a26d-585cbe4c4aaa,1105-070950-djhheq45-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1303)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-05T07:10:03.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-359b3085-b280-46e0-803e-4214a0d0f733"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-fc989ddd-3721-419a-847a-ff478003b0d1""}, statsOnLoad -> false)",null,List(2163559625184539),c39110db-b9e1-48d4-bb13-d336d23efbd3,1105-070950-djhheq45-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
select * from retail_products_sql version as of 1;

product_id,product_name,price
1,Laptop,50000
2,Mpuse,800
3,Keyboard,1500


In [0]:
%sql
select * from retail_products_sql version as of 3;

product_id,product_name,price
1,Laptop,50000
3,Keyboard,1500
2,Wireless Mouse,950
10,USB Hub,1200


In [0]:
%sql
optimize retail_products_sql;

path,metrics
abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/a5e339a0-0baf-48ce-a33a-2e669cb9c673,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793869657596, 1793869658318, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"


In [0]:
%sql
optimize retail_products_sql zorder by (product_id);

path,metrics
abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/a5e339a0-0baf-48ce-a33a-2e669cb9c673,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2202), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793869976697, 1793869977428, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"


DELTA TABLE PRACTICE EXERCISE

In [0]:
%sql
CREATE OR REPLACE TABLE courier_parcels_delta (
    parcel_id INT,
    customer_name STRING,
    destination_city STRING,
    service_type STRING,
    weight_kg DOUBLE,
    shipping_fee INT,
    delivery_status STRING
)
USING DELTA;

INSERT INTO courier_parcels_delta VALUES
(101, 'Arjun', 'Hyderabad', 'Express', 2.5, 500, 'Delivered'),
(102, 'Meera', 'Mumbai', 'Standard', 4.0, 350, 'In Transit'),
(103, 'Karan', 'Delhi', 'Express', 1.5, 450, 'Pending'),
(104, 'Sneha', 'Pune', 'Standard', 3.0, 300, 'Delivered'),
(105, 'Rohit', 'Chennai', 'Express', 5.0, 800, 'In Transit'),
(106, 'Divya', 'Hyderabad', 'Standard', 2.0, 250, 'Cancelled'),
(107, 'Nikhil', 'Bengaluru', 'Express', 6.0, 950, 'Delivered'),
(108, 'Pooja', 'Mumbai', 'Standard', 1.0, 150, 'Pending'),
(109, 'Vikram', 'Delhi', 'Express', 3.5, 650, 'In Transit'),
(110, 'Anjali', 'Pune', 'Standard', 4.5, 400, 'Cancelled'),
(111, 'Farhan', 'Chennai', 'Express', 2.0, 550, 'Delivered'),
(112, 'Kavya', 'Hyderabad', 'Standard', 5.5, 600, 'Pending'),
(113, 'Manoj', 'Bengaluru', 'Express', 1.5, 400, 'In Transit'),
(114, 'Riya', 'Delhi', 'Standard', 3.0, 300, 'Delivered'),
(115, 'Tarun', 'Mumbai', 'Express', 4.0, 700, 'Pending');

num_affected_rows,num_inserted_rows
15,15


Part A

In [0]:
%sql
SELECT * FROM courier_parcels_delta;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
101,Arjun,Hyderabad,Express,2.5,500,Delivered
102,Meera,Mumbai,Standard,4.0,350,In Transit
103,Karan,Delhi,Express,1.5,450,Pending
104,Sneha,Pune,Standard,3.0,300,Delivered
105,Rohit,Chennai,Express,5.0,800,In Transit
106,Divya,Hyderabad,Standard,2.0,250,Cancelled
107,Nikhil,Bengaluru,Express,6.0,950,Delivered
108,Pooja,Mumbai,Standard,1.0,150,Pending
109,Vikram,Delhi,Express,3.5,650,In Transit
110,Anjali,Pune,Standard,4.5,400,Cancelled


In [0]:
%sql
CREATE TABLE courier_parcels_copy USING DELTA AS 
SELECT * FROM courier_parcels_delta;

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("parcel_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("destination_city", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("shipping_fee", IntegerType(), True),
    StructField("delivery_status", StringType(), True)
])
data = [
    (119, 'Suresh', 'Kolkata', 'Express', 3.0, 500, 'Pending'),
    (120, 'Neha', 'Mumbai', 'Standard', 2.0, 300, 'In Transit'),
    (121, 'Amit', 'Delhi', 'Express', 4.5, 750, 'Delivered')
]
df = spark.createDataFrame(data, schema)
df.write.format("delta").mode("overwrite").saveAsTable("courier_parcels_pyspark")

In [0]:
%sql
DESCRIBE DETAIL courier_parcels_delta;
DESCRIBE DETAIL courier_parcels_copy;
DESCRIBE DETAIL courier_parcels_pyspark;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c0224284-f0b0-4225-9dfa-1418980d8f81,dbt.default.courier_parcels_pyspark,null,abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/c1ea17a0-3113-460f-baa9-eb12a49111f9,2026-11-05T09:30:28.672Z,2026-11-05T09:30:30.000Z,List(),List(),1,2342,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
INSERT INTO courier_parcels_delta VALUES 
(116, 'Sameer', 'Jaipur', 'Express', 2.5, 550, 'Pending');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE courier_parcels_delta SET delivery_status = 'Delivered' WHERE parcel_id = 103;

num_affected_rows
1


In [0]:
%sql
UPDATE courier_parcels_delta SET shipping_fee = shipping_fee + 100 WHERE service_type = 'Express';

num_affected_rows
9


In [0]:
%sql
UPDATE courier_parcels_delta SET destination_city = 'Hyderabad' WHERE parcel_id = 108;

num_affected_rows
1


In [0]:
%sql
DELETE FROM courier_parcels_delta WHERE delivery_status = 'Cancelled';

num_affected_rows
2


In [0]:
%sql
SELECT * FROM courier_parcels_delta ORDER BY shipping_fee DESC;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
107,Nikhil,Bengaluru,Express,6.0,1050,Delivered
105,Rohit,Chennai,Express,5.0,900,In Transit
115,Tarun,Mumbai,Express,4.0,800,Pending
109,Vikram,Delhi,Express,3.5,750,In Transit
116,Sameer,Jaipur,Express,2.5,650,Pending
111,Farhan,Chennai,Express,2.0,650,Delivered
101,Arjun,Hyderabad,Express,2.5,600,Delivered
112,Kavya,Hyderabad,Standard,5.5,600,Pending
103,Karan,Delhi,Express,1.5,550,Delivered
113,Manoj,Bengaluru,Express,1.5,500,In Transit


Part C

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parcels AS
SELECT * FROM VALUES
(102, 'Meera', 'Mumbai', 'Express', 4.0, 650, 'Delivered'),
(105, 'Rohit', 'Chennai', 'Express', 5.0, 900, 'Delivered'),
(117, 'Ayesha', 'Kolkata', 'Standard', 2.0, 300, 'Pending'),
(118, 'Rahul', 'Jaipur', 'Express', 3.0, 600, 'In Transit')
AS incoming(parcel_id, customer_name, destination_city, service_type, weight_kg, shipping_fee, delivery_status);

In [0]:
%sql
SELECT * FROM incoming_parcels;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


In [0]:
%sql
SELECT src.* FROM incoming_parcels src INNER JOIN courier_parcels_delta tgt ON src.parcel_id = tgt.parcel_id;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered


In [0]:
%sql
SELECT src.* FROM incoming_parcels src LEFT JOIN courier_parcels_delta tgt ON src.parcel_id = tgt.parcel_id WHERE tgt.parcel_id IS NULL;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


In [0]:
%sql
MERGE INTO courier_parcels_delta AS target USING incoming_parcels AS source ON target.parcel_id = source.parcel_id
WHEN MATCHED THEN  UPDATE SET *
WHEN NOT MATCHED THEN  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
SELECT * FROM courier_parcels_delta WHERE parcel_id IN (102, 105);

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered


In [0]:
%sql
SELECT * FROM courier_parcels_delta 
WHERE parcel_id IN (117, 118);

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


In [0]:
%sql
DESCRIBE DETAIL courier_parcels_delta;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,eb1a787b-1581-4bbc-a7fc-dc88d78a2923,dbt.default.courier_parcels_delta,null,abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/f2368d80-3949-4a24-ac36-38e863355e52,2026-11-05T09:24:33.256Z,2026-11-05T09:37:19.000Z,List(),List(),1,3605,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-12f5f972-8a23-46b0-9d6c-02c34f5d6d71, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-520962fe-b0c8-4923-b528-7d655b57f698)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE HISTORY courier_parcels_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:37:19.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),2b2d71b4-5d4a-400f-b47f-20c61567f2f3,1105-070950-djhheq45-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15196, p25FileSize -> 3605, numDeletionVectorsRemoved -> 1, minFileSize -> 3605, numAddedFiles -> 1, maxFileSize -> 3605, p75FileSize -> 3605, p50FileSize -> 3605, numAddedBytes -> 3605)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
9,2026-11-05T09:37:17.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(parcel_id#17777 = parcel_id#17756)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2163559625184539),2b2d71b4-5d4a-400f-b47f-20c61567f2f3,1105-070950-djhheq45-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11613, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2934, materializeSourceTimeMs -> 112, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1086, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1709)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-05T09:34:42.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),0ba3351c-14ba-4abe-b520-ea63b00d01c8,1105-070950-djhheq45-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6558, p25FileSize -> 3583, numDeletionVectorsRemoved -> 1, minFileSize -> 3583, numAddedFiles -> 1, maxFileSize -> 3583, p75FileSize -> 3583, p50FileSize -> 3583, numAddedBytes -> 3583)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-05T09:34:41.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(delivery_status#16693 = Cancelled)""])",null,List(2163559625184539),0ba3351c-14ba-4abe-b520-ea63b00d01c8,1105-070950-djhheq45-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1003, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 568, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 433)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-05T09:34:21.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#16088 = 108)""])",null,List(2163559625184539),308f0a53-a801-4f7f-922d-1f6a80c7f04f,1105-070950-djhheq45-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2355, numDeletionVectorsUpdated -> 0, scanTimeMs -> 887, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2967, rewriteTimeMs -> 1467)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-05T09:34:04.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map

In [0]:
%sql
DESCRIBE HISTORY courier_parcels_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:37:19.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),2b2d71b4-5d4a-400f-b47f-20c61567f2f3,1105-070950-djhheq45-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15196, p25FileSize -> 3605, numDeletionVectorsRemoved -> 1, minFileSize -> 3605, numAddedFiles -> 1, maxFileSize -> 3605, p75FileSize -> 3605, p50FileSize -> 3605, numAddedBytes -> 3605)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
9,2026-11-05T09:37:17.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(parcel_id#17777 = parcel_id#17756)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2163559625184539),2b2d71b4-5d4a-400f-b47f-20c61567f2f3,1105-070950-djhheq45-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11613, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2934, materializeSourceTimeMs -> 112, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1086, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1709)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-05T09:34:42.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),0ba3351c-14ba-4abe-b520-ea63b00d01c8,1105-070950-djhheq45-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6558, p25FileSize -> 3583, numDeletionVectorsRemoved -> 1, minFileSize -> 3583, numAddedFiles -> 1, maxFileSize -> 3583, p75FileSize -> 3583, p50FileSize -> 3583, numAddedBytes -> 3583)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-05T09:34:41.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(delivery_status#16693 = Cancelled)""])",null,List(2163559625184539),0ba3351c-14ba-4abe-b520-ea63b00d01c8,1105-070950-djhheq45-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1003, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 568, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 433)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-05T09:34:21.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#16088 = 108)""])",null,List(2163559625184539),308f0a53-a801-4f7f-922d-1f6a80c7f04f,1105-070950-djhheq45-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2355, numDeletionVectorsUpdated -> 0, scanTimeMs -> 887, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2967, rewriteTimeMs -> 1467)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-05T09:34:04.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map

In [0]:
%sql
SELECT * FROM courier_parcels_delta VERSION AS OF 9;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
108,Pooja,Hyderabad,Standard,1.0,150,Pending
104,Sneha,Pune,Standard,3.0,300,Delivered
112,Kavya,Hyderabad,Standard,5.5,600,Pending
114,Riya,Delhi,Standard,3.0,300,Delivered
101,Arjun,Hyderabad,Express,2.5,600,Delivered
107,Nikhil,Bengaluru,Express,6.0,1050,Delivered
109,Vikram,Delhi,Express,3.5,750,In Transit
111,Farhan,Chennai,Express,2.0,650,Delivered
113,Manoj,Bengaluru,Express,1.5,500,In Transit
115,Tarun,Mumbai,Express,4.0,800,Pending


In [0]:
%sql
SELECT * FROM courier_parcels_delta WHERE parcel_id = 102;
SELECT * FROM courier_parcels_delta VERSION AS OF 7 WHERE parcel_id = 102;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Standard,4.0,350,In Transit


Part E

In [0]:
%sql
OPTIMIZE courier_parcels_delta;

path,metrics
abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/f2368d80-3949-4a24-ac36-38e863355e52,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793871713131, 1793871713763, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL courier_parcels_delta;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,eb1a787b-1581-4bbc-a7fc-dc88d78a2923,dbt.default.courier_parcels_delta,null,abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/f2368d80-3949-4a24-ac36-38e863355e52,2026-11-05T09:24:33.256Z,2026-11-05T09:37:19.000Z,List(),List(),1,3605,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-12f5f972-8a23-46b0-9d6c-02c34f5d6d71, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-520962fe-b0c8-4923-b528-7d655b57f698)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
OPTIMIZE courier_parcels_delta ZORDER BY (destination_city);

path,metrics
abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/f2368d80-3949-4a24-ac36-38e863355e52,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3605), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793871759760, 1793871760300, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE HISTORY courier_parcels_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:37:19.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),2b2d71b4-5d4a-400f-b47f-20c61567f2f3,1105-070950-djhheq45-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15196, p25FileSize -> 3605, numDeletionVectorsRemoved -> 1, minFileSize -> 3605, numAddedFiles -> 1, maxFileSize -> 3605, p75FileSize -> 3605, p50FileSize -> 3605, numAddedBytes -> 3605)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
9,2026-11-05T09:37:17.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(parcel_id#17777 = parcel_id#17756)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2163559625184539),2b2d71b4-5d4a-400f-b47f-20c61567f2f3,1105-070950-djhheq45-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11613, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2934, materializeSourceTimeMs -> 112, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1086, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1709)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-05T09:34:42.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2163559625184539),0ba3351c-14ba-4abe-b520-ea63b00d01c8,1105-070950-djhheq45-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6558, p25FileSize -> 3583, numDeletionVectorsRemoved -> 1, minFileSize -> 3583, numAddedFiles -> 1, maxFileSize -> 3583, p75FileSize -> 3583, p50FileSize -> 3583, numAddedBytes -> 3583)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-05T09:34:41.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(delivery_status#16693 = Cancelled)""])",null,List(2163559625184539),0ba3351c-14ba-4abe-b520-ea63b00d01c8,1105-070950-djhheq45-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1003, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 568, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 433)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-05T09:34:21.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(parcel_id#16088 = 108)""])",null,List(2163559625184539),308f0a53-a801-4f7f-922d-1f6a80c7f04f,1105-070950-djhheq45-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2355, numDeletionVectorsUpdated -> 0, scanTimeMs -> 887, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2967, rewriteTimeMs -> 1467)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-05T09:34:04.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map